In [ ]:
# representations---tfidf_vectorization

In [1]:
# Load & validation

import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed/reviews_preprocessed.csv")
df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# Hard checks
assert "clean_review" in df.columns, "clean_review missing"
assert "sentiment" in df.columns, "sentiment missing"

missing_clean = df["clean_review"].isna().sum()
print("Missing clean_review:", missing_clean)

print("\nClass distribution:")
print(df["sentiment"].value_counts())

print("\nTotal samples:", len(df))

Loaded: D:\University of Bicocca\Data science\2nd year 2025-26\1st Semester\Text Mining & Search\Project\drug-review-sentiment-topic-modeling\data\processed\reviews_preprocessed.csv
Shape: (49998, 5)
Columns: ['review', 'clean_review', 'rating', 'sentiment', 'condition']
Missing clean_review: 0

Class distribution:
sentiment
1    36324
0    13674
Name: count, dtype: int64

Total samples: 49998


In [2]:
#  Build TF-IDF matrix

from sklearn.feature_extraction.text import TfidfVectorizer

X_text = df["clean_review"].astype(str).tolist()

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_tfidf = tfidf_vectorizer.fit_transform(X_text)

print("TF-IDF matrix built.")
print("Shape (n_samples, n_features):", X_tfidf.shape)

TF-IDF matrix built.
Shape (n_samples, n_features): (49998, 10000)


In [4]:
# Save TF-IDF + labels

import os
from scipy import sparse

FEATURE_DIR = Path("../data/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

tfidf_path = FEATURE_DIR / "tfidf.npz"
labels_path = FEATURE_DIR / "labels.npy"

# Save sparse matrix
sparse.save_npz(tfidf_path, X_tfidf)

# Save labels
y = df["sentiment"].values
np.save(labels_path, y)

print("Saved TF-IDF to:", tfidf_path.resolve())
print("Saved labels to:", labels_path.resolve())

Saved TF-IDF to: D:\University of Bicocca\Data science\2nd year 2025-26\1st Semester\Text Mining & Search\Project\drug-review-sentiment-topic-modeling\data\features\tfidf.npz
Saved labels to: D:\University of Bicocca\Data science\2nd year 2025-26\1st Semester\Text Mining & Search\Project\drug-review-sentiment-topic-modeling\data\features\labels.npy


In [5]:
# TF-IDF summary block + sanity checks

import numpy as np

n_samples, n_features = X_tfidf.shape
nnz = X_tfidf.nnz
sparsity = 1 - (nnz / (n_samples * n_features))

feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
idf = tfidf_vectorizer.idf_

print("===== REPRESENTATION A — TF-IDF (Sparse) =====")
print("Input column: clean_review")
print("Params: max_features=10000, ngram_range=(1,2)")
print(f"Matrix shape: ({n_samples}, {n_features})")
print(f"Non-zeros (nnz): {nnz}")
print(f"Sparsity: {sparsity:.4f}")
print("Saved:")
print(" -", tfidf_path)
print(" -", labels_path)

# Interpretability sanity: show 15 terms with highest IDF (rarest terms in vocab)
top_idf_idx = np.argsort(idf)[-15:][::-1]
print("\nTop 15 terms by IDF (rare terms):")
for t, v in zip(feature_names[top_idf_idx], idf[top_idf_idx]):
    print(f"{t:25s}  idf={v:.2f}")

print("==============================================")

===== REPRESENTATION A — TF-IDF (Sparse) =====
Input column: clean_review
Params: max_features=10000, ngram_range=(1,2)
Matrix shape: (49998, 10000)
Non-zeros (nnz): 2122355
Sparsity: 0.9958
Saved:
 - ..\data\features\tfidf.npz
 - ..\data\features\labels.npy

Top 15 terms by IDF (rare terms):
u2026                      idf=9.74
hiccup                     idf=8.99
complera                   idf=8.99
vivelle                    idf=8.93
modafinil                  idf=8.88
lice                       idf=8.82
cephalexin                 idf=8.82
parnate                    idf=8.82
repatha                    idf=8.82
xyrem                      idf=8.78
xiidra                     idf=8.78
angina                     idf=8.78
tirosint                   idf=8.78
forteo                     idf=8.73
eliquis                    idf=8.73


In [7]:
# Load Sentence-Transformers model

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"  # 384-dimensional, fast, strong baseline
model = SentenceTransformer(MODEL_NAME)

print("Loaded encoder:", MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded encoder: all-MiniLM-L6-v2


In [8]:
# Compute dense embeddings (feature extraction only)

texts = df["clean_review"].astype(str).tolist()

bert_embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

print("Embeddings computed.")
print("Embedding shape:", bert_embeddings.shape)

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Embeddings computed.
Embedding shape: (49998, 384)


In [9]:
#Save embeddings ===> Summary block

import numpy as np

bert_path = FEATURE_DIR / "bert_embeddings.npy"

np.save(bert_path, bert_embeddings)

# Sanity stats for first embedding vector
first_vec = bert_embeddings[0]
vec_min = float(first_vec.min())
vec_max = float(first_vec.max())
vec_mean = float(first_vec.mean())

print("===== REPRESENTATION B — BERT Embeddings (Dense) =====")
print("Input column: clean_review")
print("Encoder: sentence-transformers/all-MiniLM-L6-v2")
print("Fine-tuning: NO (feature extraction only)")
print(f"Embedding shape: {bert_embeddings.shape}")
print("Saved:")
print(" -", bert_path)

print("\nSanity check (first vector stats):")
print(f"Min:  {vec_min:.4f}")
print(f"Max:  {vec_max:.4f}")
print(f"Mean: {vec_mean:.4f}")

print("Embedding dimension:", bert_embeddings.shape[1])
print("=======================================================")

===== REPRESENTATION B — BERT Embeddings (Dense) =====
Input column: clean_review
Encoder: sentence-transformers/all-MiniLM-L6-v2
Fine-tuning: NO (feature extraction only)
Embedding shape: (49998, 384)
Saved:
 - ..\data\features\bert_embeddings.npy

Sanity check (first vector stats):
Min:  -0.1522
Max:  0.1573
Mean: -0.0003
Embedding dimension: 384
